<a href="https://colab.research.google.com/github/Linford24/AfricaBp_Computer_Vision_for_Biodiversity_Classification/blob/main/FinetuningMbart50.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install peft transformers bitsandbytes accelerate datasets sentencepiece evaluate sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.0 MB/s eta 0:00:00


In [ ]:
import os
import torch
from dataclasses import dataclass, field
from typing import Optional
from datasets import load_dataset
from transformers import (
    MBartForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    set_seed
)
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training
)
import numpy as np
import evaluate

In [ ]:
import requests
import json
import os
import zipfile

In [ ]:
def prepare_opus_data_split(url: str, train_path: str, eval_path: str, max_train: int = 2500, max_eval: int = 500):
    zip_path = "opus_wikimedia.zip"

    # 1. Download dataset
    print("Downloading OPUS dataset...")
    response = requests.get(url, stream=True)
    response.raise_for_status()
    with open(zip_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)

    # 2. Extract files
    print("Extracting files...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall("opus_data")

    src_file = "opus_data/wikimedia.en-tw.en"
    tgt_file = "opus_data/wikimedia.en-tw.tw"

    # 3. Read and pair all sentences
    print("Reading and pairing parallel sentences...")
    all_records = []

    with open(src_file, 'r', encoding='utf-8') as f_src, open(tgt_file, 'r', encoding='utf-8') as f_tgt:
        for src_line, tgt_line in zip(f_src, f_tgt):
            src_str = src_line.strip()
            tgt_str = tgt_line.strip()

            # Skip empty lines if any exist
            if src_str and tgt_str:
                all_records.append({
                    "translation": {
                        "en_XX": src_str,
                        "tl_XX": tgt_str
                    }
                })

    # 4. Create Train and Eval splits sequentially
    total_needed = max_train + max_eval
    if len(all_records) < total_needed:
        print(f"⚠️ Warning: Dataset only has {len(all_records)} pairs. Requested total: {total_needed}.")

    train_records = all_records[:max_train]
    eval_records = all_records[max_train : max_train + max_eval]

    # 5. Save Train Split
    with open(train_path, 'w', encoding='utf-8') as f_out:
        json.dump(train_records, f_out, ensure_ascii=False, indent=4)

    # 6. Save Validation Split
    with open(eval_path, 'w', encoding='utf-8') as f_out:
        json.dump(eval_records, f_out, ensure_ascii=False, indent=4)

    print(f"Successfully created:")
    print(f"  - Train split: {len(train_records)} samples -> {train_path}")
    print(f"  - Eval split:  {len(eval_records)} samples -> {eval_path}")

In [ ]:
@dataclass
class FineTuneConfig:
  # Model
  model_name= "masakhane/afri-mbart50"
  output_dir= "./finetuned_afri_mbart50"

  # QLoRA quantization
  load_in_4bit: bool=True
  bnb_4bit_compute_dtype: str="bfloat16"
  bnb_4bit_quant_type: str="nf4"

  # LoRA Adapter
  lora_r: int=16
  lora_alpha: int=32
  lora_dropout: float=0.05
  lora_target_modules: list = field(default_factory=lambda: [
      "q_proj", "v_proj", "k_proj", "out_proj", # attention
      "fc1", "fc2"  # FFN layers
  ])

  # Data
  dataset_name: str="text"
  train_file: Optional[str]="train_parallel.json"
  validation_file: Optional[str]="eval_parallel.json"

  # mBART-50 Specific Language Codes
  source_lang: str="en_XX"
  target_lang: str="tl_XX"


  max_source_length: int=128
  max_target_length: int=128
  max_train_samples: int=2500
  max_eval_samples: int=500


  # Training
  num_train_epochs: int=3
  per_device_train_batch_size: int=4
  per_device_eval_batch_size: int=4
  gradient_accumulation_steps: int=4
  learning_rate: float = 2e-4
  warmup_steps: int = 100
  logging_steps: int = 50
  eval_steps: int = 200
  save_steps: int = 200
  fp16: bool = False
  bf16: bool = True
  seed: int = 42


In [ ]:
# Run the preparation using your config values
config = FineTuneConfig()
config.train_file = "train_parallel.json"
config.validation_file = "eval_parallel.json"
prepare_opus_data_split(
    url="https://object.pouta.csc.fi/OPUS-wikimedia/v20230407/moses/en-tw.txt.zip",
    train_path=config.train_file,
    eval_path=config.validation_file,
    max_train=config.max_train_samples,
    max_eval=config.max_eval_samples
)

Extracting files...
Reading and pairing parallel sentences...
Successfully created:
  - Train split: 2500 samples -> train_parallel.json
  - Eval split:  500 samples -> eval_parallel.json


In [ ]:
from datasets import load_dataset
raw_datasets = load_dataset(
    "json",
    data_files={
        "train": config.train_file,
        "validation": config.validation_file
    }
)
print(raw_datasets)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 2500
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 500
    })
})


In [ ]:
# Quantization Config
compute_dtype = getattr(torch, config.bnb_4bit_compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=config.load_in_4bit,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_quant_type=config.bnb_4bit_quant_type,
    bnb_4bit_use_double_quant=True
)

print("BitsAndBytes QLoRA config ready")

BitsAndBytes QLoRA config ready


In [ ]:
# Load Tokenizer and Model
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
tokenizer = AutoTokenizer.from_pretrained(config.model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.41k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.04k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

In [ ]:
tokenizer.src_lang = config.source_lang
tokenizer.tgt_lang = config.target_lang

In [ ]:
print(f"✅  Tokenizer loaded  (src={config.source_lang}, tgt={config.target_lang})")


✅  Tokenizer loaded  (src=en_XX, tgt=tl_XX)


In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(
    config.model_name,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=compute_dtype
)

pytorch_model.bin:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/519 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [ ]:
# Load Model in 4-bit QLoRA

In [ ]:
model = prepare_model_for_kbit_training(model)

In [ ]:
# LoRA Adapter

lora_config = LoraConfig(
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    target_modules=config.lora_target_modules,
    lora_dropout=config.lora_dropout,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
    inference_mode=False
)

In [ ]:
# 2. Disable caching during training (Fixes the "use_cache" conflict)
model.config.use_cache = False

In [ ]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("LoRA Adapters injected")

trainable params: 8,650,752 || all params: 1,387,696,128 || trainable%: 0.6234
LoRA Adapters injected


In [ ]:
# 1. Ensure the model recognizes it's in training mode
model.train()

# 2. Force the input embedding layer to require gradients
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()
else:
    model.get_input_embeddings().requires_grad_ = True

# 3. Double-check and force ALL LoRA parameters to require gradients
trainable_params = 0
all_param = 0
for name, param in model.named_parameters():
    all_param += param.numel()
    # If it belongs to LoRA, force it to adapt!
    if "lora_" in name:
        param.requires_grad = True
        trainable_params += param.numel()

print(f"🔒 Gradient Check: Trainable params: {trainable_params:,} || All params: {all_param:,}")

🔒 Gradient Check: Trainable params: 8,650,752 || All params: 1,211,535,360


In [ ]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 2500
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 500
    })
})

In [ ]:
def preprocess(examples):
  inputs = [ex["en_XX"] for ex in examples["translation"]]
  targets = [ex["tl_XX"] for ex in examples["translation"]]


  model_inputs = tokenizer(
      inputs,
      text_target=targets,
      max_length=config.max_source_length,
      max_target_length=config.max_target_length,
      truncation=True,
      padding="max_length"
  )

  # Padding logic
  model_inputs["labels"] = [
      [(t if t != tokenizer.pad_token_id else -100) for t in label]
      for label in model_inputs["labels"]
  ]

  return model_inputs

In [ ]:
train_dataset = raw_datasets["train"]
eval_dataset  = raw_datasets["validation"]

if config.max_train_samples:
    train_dataset = train_dataset.select(range(min(config.max_train_samples, len(train_dataset))))
if config.max_eval_samples:
    eval_dataset = eval_dataset.select(range(min(config.max_eval_samples, len(eval_dataset))))

train_dataset = train_dataset.map(preprocess, batched=True, remove_columns=train_dataset.column_names)
eval_dataset  = eval_dataset.map(preprocess,  batched=True, remove_columns=eval_dataset.column_names)
print(f"✅  Dataset ready  train={len(train_dataset)}  eval={len(eval_dataset)}")

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

✅  Dataset ready  train=2500  eval=500


In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=config.output_dir,
    num_train_epochs=config.num_train_epochs,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=config.learning_rate,
    warmup_steps=config.warmup_steps,
    fp16=config.fp16,
    bf16=config.bf16,
    logging_steps=config.logging_steps,
    eval_strategy="steps",
    eval_steps=config.eval_steps,
    save_strategy="steps",
    save_steps=config.save_steps,
    save_total_limit=2,
    load_best_model_at_end=False,
    predict_with_generate=True,
    generation_max_length=config.max_target_length,
    report_to="none",
    seed=config.seed,
    dataloader_num_workers=2,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer, model=model, label_pad_token_id=-100, pad_to_multiple_of=8
)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=None
)

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
print(f"Starting QLoRA + LoRA Finetuning")
trainer.train()

Starting QLoRA + LoRA Finetuning


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
200,28.055544,3.369715
400,23.918330,3.074864


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=471, training_loss=32.515147150448676, metrics={'train_runtime': 2606.1525, 'train_samples_per_second': 2.878, 'train_steps_per_second': 0.181, 'total_flos': 3556390993920000.0, 'train_loss': 32.515147150448676, 'epoch': 3.0})

In [ ]:
#import gc

#gc.collect()
#torch.cuda.empty_cache()

In [ ]:
model.save_pretrained(config.output_dir)
tokenizer.save_pretrained(config.output_dir)
print(f"✅  LoRA adapters saved to  {config.output_dir}")

✅  LoRA adapters saved to  ./finetuned_afri_mbart50


In [ ]:
def translate(text: str, src_lang: str = config.source_lang, tgt_lang: str = config.target_lang) -> str:
    tokenizer.src_lang = src_lang
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    forced_bos = tokenizer.convert_tokens_to_ids(tgt_lang)

    # CRITICAL FOR INFERENCE: Temporarily re-enable cache for generation stability
    model.config.use_cache = True

    with torch.no_grad():
        out = model.generate(
            **inputs,
            forced_bos_token_id=forced_bos,
            max_new_tokens=128,
            num_beams=4,
            early_stopping=True
        )

    # Set it back to False just in case you resume training later
    model.config.use_cache = False
    return tokenizer.decode(out[0], skip_special_tokens=True)

In [ ]:
sample = "Hello my friend, how has your day been?"
print(f"\n📝 Sample translation:\n  EN: {sample}\n  SW: {translate(sample)}")

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
`use_cache=True` is incompatible with gradient checkpointing`. Setting `use_cache=False`...



📝 Sample translation:
  EN: Hello my friend, how has your day been?
  SW: nn nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh nh
